In [3]:
# Step 1: Starbucks Reviews Data Preprocessing
# Fixed for: ValueError + Tokenizer Warning + 3-class labels

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from transformers import pipeline
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("reviews_data.csv")

# Keep only review column
df_clean = df[["Review"]].copy()
df_clean = df_clean.dropna(subset=["Review"])

print("Cleaned data shape:", df_clean.shape)
print(df_clean.head())

# Load sentiment model
classifier = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

# Function to create 3-class labels: Positive, Negative, Neutral
def get_sentiment(text):
    text = str(text)[:512]
    res = classifier(text)[0]
    if res["label"] == "POSITIVE" and res["score"] > 0.7:
        return "Positive"
    elif res["label"] == "NEGATIVE" and res["score"] > 0.7:
        return "Negative"
    else:
        return "Neutral"

df_clean["sentiment"] = df_clean["Review"].apply(get_sentiment)

print("\nSentiment distribution:")
print(df_clean["sentiment"].value_counts())

# ---------------- FIXED PART ----------------
# Split without stratify to avoid class size error
train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

# Save final dataset files
train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv", index=False)
test_df.to_csv("test.csv", index=False)

print("\n✅ Done! Files saved:")
print("train.csv, val.csv, test.csv")

Cleaned data shape: (850, 1)
                                              Review
0  Amber and LaDonna at the Starbucks on Southwes...
1  ** at the Starbucks by the fire station on 436...
2  I just wanted to go out of my way to recognize...
3  Me and my friend were at Starbucks and my card...
4  I’m on this kick of drinking 5 cups of warm wa...


Device set to use mps:0



Sentiment distribution:
sentiment
Negative    697
Positive    139
Neutral      14
Name: count, dtype: int64

✅ Done! Files saved:
train.csv, val.csv, test.csv


In [6]:
# Step 2: Fine-Tune BERT Model for Starbucks Sentiment Analysis
# WITH Training Loss (added logging_strategy)

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install transformers datasets pandas torch --quiet

import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

# 1. Load Dataset
train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")

label2id = {"Positive": 0, "Negative": 1, "Neutral": 2}
id2label = {0: "Positive", 1: "Negative", 2: "Neutral"}

train_df["label"] = train_df["sentiment"].map(label2id)
val_df["label"] = val_df["sentiment"].map(label2id)
test_df["label"] = test_df["sentiment"].map(label2id)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

# 2. Model & Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=3, id2label=id2label, label2id=label2id
)

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["Review"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 3. Training Arguments (ADDED logging_strategy TO SHOW TRAIN LOSS)
training_args = TrainingArguments(
    output_dir="starbucks_sentiment_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    overwrite_output_dir=True,
    logging_strategy="epoch"   # ✅ THIS LINE MAKES TRAIN LOSS APPEAR
)

# 4. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

# Start training
trainer.train()

# Save model
model.save_pretrained("final_starbucks_model")
tokenizer.save_pretrained("final_starbucks_model")

# Test
classifier = pipeline("text-classification", model="final_starbucks_model")
print("Test result:", classifier("Great coffee and friendly service"))

print("\n✅ Step 2 FINISHED successfully!")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/680 [00:00<?, ? examples/s]

Map:   0%|          | 0/85 [00:00<?, ? examples/s]

/Users/zhaoruoxi/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.470300,0.198383
2,0.241100,0.159230
3,0.131700,0.187011


/Users/zhaoruoxi/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/zhaoruoxi/Library/Python/3.9/lib/python/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Device set to use mps:0


Test result: [{'label': 'Negative', 'score': 0.5871835947036743}]

✅ Step 2 FINISHED successfully!


In [43]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

# 情感分析（保持你原来的）
sentiment_analyzer = pipeline("text-classification", model="final_starbucks_model", device=-1)

# 🔥 更强模型：LaMini-Flan-T5-248M（CPU 可跑、客服/摘要更好）
model_name = "MBZUAI/LaMini-Flan-T5-248M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 评论
review = """I really enjoy the rich coffee flavor and cozy atmosphere at this Starbucks. I am quite disappointed with the extremely long waiting time during busy hours. I hope the store can improve service efficiency and shorten customer waiting time soon"""

# 情感
sentiment = sentiment_analyzer(review)[0]["label"]

# 摘要（更准、更短）
summary_prompt = f"summarize customer review in one short sentence: {review}"
summary_ids = model.generate(
    **tokenizer(summary_prompt, return_tensors="pt", truncation=True, max_length=512),
    max_length=30,
    min_length=8,
    num_beams=4,
    do_sample=False,
    repetition_penalty=1.2
)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# 客服回复（更礼貌、专业）
reply_prompt = (
    f"Customer feedback summary: {summary}\n"
    "Write a polite Starbucks customer service reply starting with: Thank you for your valuable feedback."
)
reply_ids = model.generate(
    **tokenizer(reply_prompt, return_tensors="pt", truncation=True, max_length=512),
    max_length=50,
    min_length=20,
    num_beams=5,
    do_sample=False,
    repetition_penalty=1.2
)
reply = tokenizer.decode(reply_ids[0], skip_special_tokens=True)

# 输出
print("=== Customer Review ===")
print(review)
print("\n=== Sentiment ===")
print(sentiment)
print("\n=== Summary ===")
print(summary)
print("\n=== Generated Dynamic Reply ===")
print(reply)

Device set to use cpu


=== Customer Review ===
I really enjoy the rich coffee flavor and cozy atmosphere at this Starbucks. I am quite disappointed with the extremely long waiting time during busy hours. I hope the store can improve service efficiency and shorten customer waiting time soon

=== Sentiment ===
Positive

=== Summary ===
The customer enjoyed the rich coffee flavor and cozy atmosphere at Starbucks, but was disappointed with the long waiting time during busy hours.

=== Generated Dynamic Reply ===
Thank you for your feedback, I appreciate it. However, I am disappointed with the long waiting time during busy hours at Starbucks. Please let me know if there is anything else I can assist you with. Best regards, [Your Name]
